# 02 — Baseline Attribution Graphs (base model)

Runs attribution graph generation on **base Gemma-2-2B** (no fine-tuning)
over the **full generated dataset** (target: 300 prompts, analysis-only usage). Computes structural
statistics for each prompt and saves them to `results/statistics/stats_base.json`.

**Run as a script** to avoid kernel timeouts:
```bash
jupyter nbconvert --to script 02_baseline_attribution.ipynb
python 02_baseline_attribution.py
```

Saves incrementally (every prompt) so a restart resumes from the last checkpoint.

**Prerequisites:** `01_dataset_generation.ipynb` must have been run.

## 0 — Environment Setup

In [1]:
import os
import sys
from pathlib import Path

IN_RUNPOD = os.environ.get("RUNPOD_POD_ID") is not None

def _find_repo_root():
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "circuit_tracer" / "__init__.py").is_file():
            return directory
    workspace = Path("/workspace")
    if workspace.is_dir():
        for child in workspace.iterdir():
            if child.is_dir() and (child / "circuit_tracer" / "__init__.py").is_file():
                return child
    repo_override = os.environ.get("CT_REPO_DIR")
    if repo_override:
        override_path = Path(repo_override).expanduser().resolve()
        if (override_path / "circuit_tracer" / "__init__.py").is_file():
            return override_path
    return None

_root = _find_repo_root()
if _root is not None:
    if str(_root) not in sys.path:
        sys.path.insert(0, str(_root))
    _my_work = _root / "my_work"
    if str(_my_work) not in sys.path:
        sys.path.insert(0, str(_my_work))
    print(f"Repo root: {_root}")
else:
    print("WARNING: could not locate circuit_tracer repo.")

try:
    from dotenv import load_dotenv
    if _root is not None and (_root / ".env").is_file():
        load_dotenv(_root / ".env")
        print("Loaded .env")
except ImportError:
    pass

# Persistent HF cache for RunPod
if IN_RUNPOD:
    persistent_root = Path(os.environ.get("RUNPOD_PERSISTENT_ROOT", "/workspace")).resolve()
    hf_home = persistent_root / "hf"
    for key, path in {
        "HF_HOME": hf_home,
        "HUGGINGFACE_HUB_CACHE": hf_home / "hub",
        "TRANSFORMERS_CACHE": hf_home / "transformers",
    }.items():
        path.mkdir(parents=True, exist_ok=True)
        os.environ[key] = str(path)
    print(f"RunPod cache: {hf_home}")

MY_WORK = _my_work if _root else Path(".").resolve()
print(f"MY_WORK: {MY_WORK}")

Repo root: /workspace/thesis_circuit_breaker
RunPod cache: /workspace/hf
MY_WORK: /workspace/thesis_circuit_breaker/my_work


## 1 — Imports & constants

In [2]:
import json
import time
import traceback

import torch
import numpy as np

# ── Plan constants (never change) ──────────────────────────────────────────────
MODEL_NAME      = "google/gemma-2-2b"
TRANSCODER_NAME = "gemma"
TOKEN_TRUE      = " True"
TOKEN_FALSE     = " False"
VOCAB_ID_TRUE   = 5569
VOCAB_ID_FALSE  = 7662

ATTR_KWARGS = dict(
    batch_size=256,
    max_feature_nodes=8192,
    offload="disk",
    verbose=False,
)

PHASE = "base"

# ── Paths ──────────────────────────────────────────────────────────────────────
DATASET_FILE   = MY_WORK / "data" / "prompts_triangle.jsonl"
GRAPHS_DIR     = MY_WORK / "results" / "graphs_base"
STATS_FILE     = MY_WORK / "results" / "statistics" / "stats_base.json"

GRAPHS_DIR.mkdir(parents=True, exist_ok=True)
STATS_FILE.parent.mkdir(parents=True, exist_ok=True)

print(f"Dataset file  : {DATASET_FILE}")
print(f"Graphs dir    : {GRAPHS_DIR}")
print(f"Stats output  : {STATS_FILE}")
print(f"CUDA available: {torch.cuda.is_available()}")

# ReplacementModel.from_pretrained expects a torch.device, not a plain string.
device_str = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(device_str)
print(f"Device        : {device}")

Dataset file  : /workspace/thesis_circuit_breaker/my_work/data/prompts_triangle.jsonl
Graphs dir    : /workspace/thesis_circuit_breaker/my_work/results/graphs_base
Stats output  : /workspace/thesis_circuit_breaker/my_work/results/statistics/stats_base.json
CUDA available: True
Device        : cuda


## 2 — Load full dataset (300 prompts)

In [9]:
analysis_prompts = []
with open(DATASET_FILE, "r", encoding="utf-8") as f:
    for line in f:
        analysis_prompts.append(json.loads(line.strip()))

binary = [p for p in analysis_prompts if p.get("task_type", "binary") == "binary"]
numeric = [p for p in analysis_prompts if p.get("task_type", "binary") == "numeric"]

print(f"Loaded {len(analysis_prompts)} prompts from full dataset")
print(f"  Binary : {len(binary)}")
print(f"    True : {sum(1 for p in binary if p['label'])}")
print(f"    False: {sum(1 for p in binary if not p['label'])}")
print(f"  Numeric: {len(numeric)}")
if numeric:
    print(f"    Targets: {sorted(set(p['label_token'] for p in numeric))}")
else:
    print("    Targets: []")

Loaded 300 prompts from full dataset
  Binary : 240
    True : 120
    False: 120
  Numeric: 60
    Targets: ['9']


## 3 — Load model

In [5]:
from circuit_tracer import ReplacementModel

model = ReplacementModel.from_pretrained(
    MODEL_NAME,
    TRANSCODER_NAME,
    dtype=torch.bfloat16,
    backend="transformerlens",
    device=device,
    lazy_encoder=True,
    lazy_decoder=True,
)
tokenizer = model.tokenizer

# Verify token IDs match constants
id_true  = tokenizer.encode(TOKEN_TRUE,  add_special_tokens=False)[-1]
id_false = tokenizer.encode(TOKEN_FALSE, add_special_tokens=False)[-1]
assert id_true  == VOCAB_ID_TRUE,  f"TOKEN_TRUE id mismatch: {id_true} != {VOCAB_ID_TRUE}"
assert id_false == VOCAB_ID_FALSE, f"TOKEN_FALSE id mismatch: {id_false} != {VOCAB_ID_FALSE}"
print(f"Token IDs verified: True={id_true}, False={id_false}")

/workspace/venvs/ct/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer
Token IDs verified: True=5569, False=7662


## 4 — First-token accuracy check (pre-attribution)

Quick sanity check: does the base model's top-1 token match the expected next token?
For binary prompts this is True/False accuracy; for numeric prompts this is exact token match.

In [15]:
overall_correct = 0

binary_correct = 0
binary_total = 0
correct_true = 0
total_true = 0
correct_false = 0
total_false = 0

numeric_correct = 0
numeric_total = 0

# Keep per-prompt predictions in memory for downstream inspection/export.
first_token_predictions = []

progress_every = 10
acc_t0 = time.time()

with torch.inference_mode():
    for idx, entry in enumerate(analysis_prompts, start=1):
        # ReplacementModel (TransformerLens backend) does not expose .model;
        # use the no-op intervention path to get baseline logits.
        logits, _ = model.feature_intervention(entry["prompt"], [])
        last_logit = logits.squeeze(0)[-1, :]
        pred_id = int(last_logit.argmax().item())
        pred_token = tokenizer.decode([pred_id])

        task_type = entry.get("task_type", "binary")
        if task_type == "binary":
            label_id = VOCAB_ID_TRUE if entry["label"] else VOCAB_ID_FALSE
            expected_response = TOKEN_TRUE if entry["label"] else TOKEN_FALSE
            binary_total += 1
            is_correct = (pred_id == label_id)
            if is_correct:
                binary_correct += 1
                overall_correct += 1
            if entry["label"]:
                total_true += 1
                if pred_id == VOCAB_ID_TRUE:
                    correct_true += 1
            else:
                total_false += 1
                if pred_id == VOCAB_ID_FALSE:
                    correct_false += 1
        else:
            # Robust handling: if label_token is multi-token (e.g. ' 9'),
            # fallback to stripped variant (e.g. '9') when single-token.
            raw_label = entry["label_token"]
            label_ids = tokenizer.encode(raw_label, add_special_tokens=False)
            expected_response = raw_label
            if len(label_ids) != 1:
                alt = raw_label.lstrip()
                alt_ids = tokenizer.encode(alt, add_special_tokens=False)
                if len(alt_ids) == 1:
                    label_ids = alt_ids
                    expected_response = alt
                else:
                    raise ValueError(
                        f"Numeric label token must resolve to a single token, got {raw_label!r} -> {label_ids}, alt {alt!r} -> {alt_ids}"
                    )
            label_id = int(label_ids[0])
            numeric_total += 1
            is_correct = (pred_id == label_id)
            if is_correct:
                numeric_correct += 1
                overall_correct += 1

        first_token_predictions.append({
            "prompt_id": entry["prompt_id"],
            "task_type": task_type,
            "expected_response": expected_response,
            "argmax_response": pred_token,
            "expected_token_id": label_id,
            "argmax_token_id": pred_id,
            "is_correct": is_correct,
        })

        if idx % progress_every == 0 or idx == len(analysis_prompts):
            elapsed = max(time.time() - acc_t0, 1e-9)
            speed = idx / elapsed
            eta = (len(analysis_prompts) - idx) / speed if speed > 0 else float("inf")
            print(
                f"[accuracy progress] {idx}/{len(analysis_prompts)} "
                f"({idx/len(analysis_prompts):.0%}) | "
                f"running overall={overall_correct/max(idx,1):.1%} | "
                f"{speed:.2f} prompts/s | ETA {eta:.1f}s"
            )

n = len(analysis_prompts)
print("Base model first-token accuracy on full dataset:")
print(f"  Overall : {overall_correct}/{n} = {overall_correct/n:.1%}")
if binary_total:
    print(f"  Binary  : {binary_correct}/{binary_total} = {binary_correct/binary_total:.1%}")
    print(f"    True  : {correct_true}/{total_true} = {correct_true/max(total_true,1):.1%}")
    print(f"    False : {correct_false}/{total_false} = {correct_false/max(total_false,1):.1%}")
if numeric_total:
    print(f"  Numeric : {numeric_correct}/{numeric_total} = {numeric_correct/numeric_total:.1%}")

if overall_correct / n > 0.80:
    print()
    print("NOTE: First-token accuracy >80%. Interpretability gains may appear smaller.")

print()
print(f"Stored per-prompt predictions in memory: first_token_predictions (n={len(first_token_predictions)})")
print("Preview (first 5):")
for row in first_token_predictions[:5]:
    print(
        f"  {row['prompt_id']}: expected={row['expected_response']!r} "
        f"argmax={row['argmax_response']!r} correct={row['is_correct']}"
    )

[accuracy progress] 10/300 (3%) | running overall=20.0% | 0.42 prompts/s | ETA 685.1s
[accuracy progress] 20/300 (7%) | running overall=40.0% | 0.53 prompts/s | ETA 523.7s
[accuracy progress] 30/300 (10%) | running overall=40.0% | 0.59 prompts/s | ETA 457.1s
[accuracy progress] 40/300 (13%) | running overall=42.5% | 0.62 prompts/s | ETA 416.9s
[accuracy progress] 50/300 (17%) | running overall=44.0% | 0.65 prompts/s | ETA 386.4s
[accuracy progress] 60/300 (20%) | running overall=45.0% | 0.66 prompts/s | ETA 363.0s
[accuracy progress] 70/300 (23%) | running overall=42.9% | 0.67 prompts/s | ETA 341.6s
[accuracy progress] 80/300 (27%) | running overall=40.0% | 0.69 prompts/s | ETA 320.3s
[accuracy progress] 90/300 (30%) | running overall=42.2% | 0.70 prompts/s | ETA 300.7s
[accuracy progress] 100/300 (33%) | running overall=42.0% | 0.71 prompts/s | ETA 282.8s
[accuracy progress] 110/300 (37%) | running overall=42.7% | 0.71 prompts/s | ETA 265.8s
[accuracy progress] 120/300 (40%) | running

This is interesting, and may cause some trouble. The " 9" is tokenized to two, so we try to get the "9" token prediction. For the numeric, it appears that the most likely first token was always " " whitespace.

In [20]:
import pandas as pd
first_token_predictions_df = pd.DataFrame(first_token_predictions)
first_token_predictions_df[first_token_predictions_df['task_type'] == 'numeric'].head()

,prompt_id,task_type,expected_response,argmax_response,expected_token_id,argmax_token_id,is_correct
3,tri_004,numeric,9,,235315,235248,False
7,tri_008,numeric,9,,235315,235248,False
8,tri_009,numeric,9,,235315,235248,False
9,tri_010,numeric,9,,235315,235248,False
11,tri_012,numeric,9,,235315,235248,False


## 5 — Attribution loop

Runs attribution for each prompt in the full dataset. Saves incrementally.

- Already-computed prompt IDs are skipped (resume support).
- Graph files are saved per-prompt under `results/graphs_base/`.
- Statistics are appended to `stats_base.json` after every prompt.
- Binary prompts use `[" True", " False"]`; numeric prompts use a single-token numeric target.
- Minimum success rate: 70% required to proceed to Stage 3.

In [27]:
from pathlib import Path
p = STATS_FILE
print(p, p.exists(), p.stat().st_size if p.exists() else None)
print(p.read_text(encoding="utf-8")[:300] if p.exists() else "missing")

/workspace/thesis_circuit_breaker/my_work/results/statistics/stats_base.json True 0



In [29]:
import utils.graph_statistics as gs
from pathlib import Path

p = Path(gs.__file__)
print("loaded module file:", p)

src = p.read_text(encoding="utf-8")
print("has robust empty-file guard:", "if not raw.strip():" in src)
print("still has old json.load(f):", "data = json.load(f)" in src)

loaded module file: /workspace/thesis_circuit_breaker/my_work/utils/graph_statistics.py
has robust empty-file guard: False
still has old json.load(f): True


In [26]:
from circuit_tracer import attribute
from circuit_tracer.utils import create_graph_files

from utils.graph_statistics import (
    compute_statistics,
    load_statistics,
    append_statistic,
)

# ── Resume: load already-computed entries ──────────────────────────────────────
done_ids: set[str] = set()
if STATS_FILE.exists():
    existing = load_statistics(STATS_FILE)
    done_ids = {e["prompt_id"] for e in existing}
    print(f"Resuming: {len(done_ids)} prompts already done.")

remaining = [p for p in analysis_prompts if p["prompt_id"] not in done_ids]
print(f"Remaining: {len(remaining)} prompts")

# ── Attribution loop ───────────────────────────────────────────────────────────
t0 = time.time()
n_success = 0
n_fail = 0

progress_every = 5
for i, entry in enumerate(remaining, start=1):
    pid = entry["prompt_id"]
    task_type = entry.get("task_type", "binary")
    print(f"[{i}/{len(remaining)}] {pid} ({task_type}) ...", end=" ", flush=True)

    # Binary prompts: contrastive attribution target
    # Numeric prompts: single-token numeric target (with robust normalization)
    if task_type == "binary":
        attr_targets = [TOKEN_TRUE, TOKEN_FALSE]
    else:
        raw_label = entry["label_token"]
        ids_raw = tokenizer.encode(raw_label, add_special_tokens=False)
        if len(ids_raw) == 1:
            target_tok = raw_label
        else:
            alt = raw_label.lstrip()
            ids_alt = tokenizer.encode(alt, add_special_tokens=False)
            if len(ids_alt) == 1:
                target_tok = alt
            else:
                raise ValueError(
                    f"Numeric attribution target must be single-token, got {raw_label!r}->{ids_raw}, alt {alt!r}->{ids_alt}"
                )
        attr_targets = [target_tok]

    try:
        graph = attribute(
            prompt=entry["prompt"],
            model=model,
            attribution_targets=attr_targets,
            **ATTR_KWARGS,
        )

        # Save graph files
        graph_out = GRAPHS_DIR / pid
        graph_out.mkdir(parents=True, exist_ok=True)
        create_graph_files(graph, slug=pid, output_path=str(graph_out))

        # Compute statistics (superset: includes legacy + supervisor metrics)
        stat = compute_statistics(graph, entry, phase=PHASE)
        n_success += 1
        density = stat.get("edge_density")
        t20 = stat.get("topk20") or {}
        score_total = t20.get("score_total")
        score_gini  = t20.get("score_gini")
        ls = stat.get("layer_stats") or {}
        ent_bits = ls.get("entropy_bits")
        n_prune_pts = len(stat.get("prune_curve") or [])
        print(
            f"OK  n_active={stat['n_active_features']}"
            f"  density={density:.4f}" if density is not None else "  density=N/A",
            f"  topk20_total={score_total:.3f}" if score_total is not None else "  topk20_total=N/A",
            f"  gini={score_gini:.3f}" if score_gini is not None else "  gini=N/A",
            f"  layer_ent={ent_bits:.2f}bits" if ent_bits is not None else "  layer_ent=N/A",
            f"  prune_curve={n_prune_pts}pts",
        )

    except Exception as exc:
        stat = {
            "prompt_id": pid,
            "phase": PHASE,
            "task_type": task_type,
            "label": entry["label"],
            "label_token": entry["label_token"],
            "template_id": entry["template_id"],
            "attribution_succeeded": False,
            "_error": str(exc),
        }
        n_fail += 1
        print(f"FAIL {exc}")

    append_statistic(stat, STATS_FILE)

    if i % progress_every == 0 or i == len(remaining):
        elapsed = max(time.time() - t0, 1e-9)
        speed = i / elapsed
        eta = (len(remaining) - i) / speed if speed > 0 else float("inf")
        running_success = n_success / max(i, 1)
        print(
            f"[attribution progress] {i}/{len(remaining)} "
            f"({i/len(remaining):.0%}) | "
            f"success={running_success:.1%} | "
            f"{speed:.3f} prompts/s | ETA {eta/60:.1f} min"
        )

elapsed = time.time() - t0
n_total = len(analysis_prompts)
n_done  = n_success + n_fail + len(done_ids)
success_rate = n_success / max(n_total - len(done_ids), 1)

print()
print(f"Done. Elapsed: {elapsed:.0f}s")
print(f"  New successes : {n_success}")
print(f"  New failures  : {n_fail}")
print(f"  Session success rate: {success_rate:.1%}")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

## 6 — Summary and go/no-go check

In [23]:
from utils.graph_statistics import load_statistics, aggregate_statistics
import json

all_stats = load_statistics(STATS_FILE)
agg = aggregate_statistics(all_stats)

print(f"Total prompts in stats file : {agg['n_total']}")
print(f"Successfully attributed     : {agg['n_succeeded']}")
print(f"Success rate                : {agg['success_rate']:.1%}")
print()

# ── Legacy scalar block ────────────────────────────────────────────────────────
legacy_metrics = [
    'n_active_features', 'edge_density', 'mean_top50_score',
    'top10_over_top50', 'layer_entropy', 'mean_error_node_weight',
    'logit_gap',
]
print("── Legacy metrics ─────────────────────────────────────────────────────")
print(f"{'Metric':<32} {'Mean':>10} {'Std':>10} {'Median':>10} {'IQR':>10}")
print("-" * 76)
for m in legacy_metrics:
    v = agg.get(m)
    if v:
        print(f"{m:<32} {v['mean']:>10.4f} {v['std']:>10.4f} {v['median']:>10.4f} {v['iqr']:>10.4f}")
    else:
        print(f"{m:<32} {'N/A':>10}")

print()

# ── Supervisor layer_stats block ───────────────────────────────────────────────
print("── Supervisor: layer scalar summary ───────────────────────────────────")
print(f"{'Metric':<32} {'Mean':>10} {'Std':>10} {'Median':>10} {'IQR':>10}")
print("-" * 76)
for m in ['layer_stats_mean', 'layer_stats_std', 'layer_stats_median', 'layer_stats_entropy_bits']:
    v = agg.get(m)
    if v:
        print(f"{m:<32} {v['mean']:>10.4f} {v['std']:>10.4f} {v['median']:>10.4f} {v['iqr']:>10.4f}")
    else:
        print(f"{m:<32} {'N/A':>10}")

print()

# ── Supervisor topk20 scalars ──────────────────────────────────────────────────
print("── Supervisor: top-K=20 influence scalars ──────────────────────────────")
for m in ['topk20_score_total', 'topk20_score_gini']:
    v = agg.get(m)
    if v:
        print(f"{m:<32} mean={v['mean']:.4f}  std={v['std']:.4f}  IQR={v['iqr']:.4f}")
    else:
        print(f"{m:<32} N/A")

print()

# ── Supervisor: prune_curve summary (mean node/edge counts per threshold) ──────
pc = agg.get("prune_curve_agg")
if pc:
    print("── Supervisor: prune curve (mean across prompts) ───────────────────────")
    print(f"{'Threshold':>10} {'Mean nodes':>12} {'Mean edges':>12} {'Mean density':>14}")
    print("-" * 52)
    for t_str, vals in sorted(pc.items(), key=lambda x: float(x[0])):
        mn = vals["mean_n_nodes"]
        me = vals["mean_n_edges"]
        md = vals["mean_density"]
        print(
            f"{float(t_str):>10.2f} "
            f"{mn if mn is None else f'{mn:>12.1f}'} "
            f"{me if me is None else f'{me:>12.1f}'} "
            f"{md if md is None else f'{md:>14.6f}'}"
        )
    print()

# ── Go/no-go check ─────────────────────────────────────────────────────────────
sr = agg['success_rate']
if sr < 0.70:
    print("STOP: Success rate below 70%. Investigate prompt set before proceeding.")
elif sr < 0.85:
    print(f"NOTE: Success rate {sr:.1%} is below 85%. Proceed but flag as limitation.")
else:
    print(f"Success rate {sr:.1%} — OK to proceed to notebook 03.")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)